# RoBERTa IMDb validation/test comparison
Upload `roberta-imdb-data.zip` through the Files sidebar. 63,836 validation and 64,405 test reviews, exact original MiniLM split/order. GPU inference only, no purchases or fine-tuning. Threshold chosen on validation and saved before test inference. External model training overlap unknown; scores are not proof of independence. Download result ZIP before runtime deletion.


In [ ]:
import base64,hashlib,zipfile,subprocess,sys,json,shutil
from pathlib import Path
from IPython.display import display,HTML
ROOT=Path('/content/roberta-imdb-v1');ROOT.mkdir(exist_ok=True)
package=Path('/content/roberta-imdb-data.zip')
assert package.exists(), 'Upload roberta-imdb-data.zip using Files sidebar first'
assert hashlib.sha256(package.read_bytes()).hexdigest()=='825ad8e2efc1fa27c18283c1c6a788f147ee6cffe8a0e792bac93db3c71f24de', 'Data ZIP hash mismatch'
with zipfile.ZipFile(package) as z:
    assert all((ROOT/item.filename).resolve().is_relative_to(ROOT.resolve()) for item in z.infolist())
    z.extractall(ROOT)
code=base64.b64decode('...')
assert hashlib.sha256(code).hexdigest()=='5719872878fac76405ec18ce94ebf490489a13a2317da9b5dc565540e238c40c'
(ROOT/'evaluate_roberta_imdb.py').write_bytes(code)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','safetensors>=0.4.3','scikit-learn'])
import torch
assert torch.cuda.is_available(), 'GPU required; no local/CPU fallback'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
OUT=ROOT/'results';OUT.mkdir(exist_ok=True)
command=[sys.executable,'-u',str(ROOT/'evaluate_roberta_imdb.py'),'--data-dir',str(ROOT),'--out',str(OUT),'--batch-size','16']
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for line in process.stdout:print(line,end='',flush=True)
status=process.wait()
archive=Path(shutil.make_archive('/content/roberta-imdb-results','zip',OUT))
for path in sorted(OUT.glob('*.json'))+[archive]:
    mime='application/zip' if path.suffix=='.zip' else 'application/json'
    encoded=base64.b64encode(path.read_bytes()).decode()
    display(HTML(f'<a download="{path.name}" href="data:{mime};base64,{encoded}">Download IMDb {path.name}</a>'))
assert status==0, 'Evaluation failed; download checkpoint files above for resume'
print('ROBERTA IMDB EVALUATION COMPLETE',flush=True)
